# 01 — Exploratory Data Analysis (RAW data)

In [ ]:
import sys
from pathlib import Path

# Resolve repo root from CWD so this cell works on any machine / CI checkout.
_here = Path.cwd().resolve()
PROJECT_ROOT = _here if (_here / "src" / "config.py").exists() else _here.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src import config
from src.utils.raw_io import load_raw_snapshot

sns.set_theme(style="whitegrid")
%matplotlib inline

df = load_raw_snapshot()
df["month"] = df["timestamp"].dt.month
df["is_smog_season"] = df["month"].isin(config.SMOG_SEASON_MONTHS).astype(int)

print(df.shape)
print(f"Range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
df.head()


## Load raw merged snapshot

In [ ]:
# Data already loaded in the PREVIOUS code cell (imports + load).
# If df is missing, re-run the first code cell only.
print("df ready:", "df" in dir())
print(df.shape if "df" in dir() else "Re-run the first code cell")


## Structure check

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nDuplicate timestamps:', df['timestamp'].duplicated().sum())
df.describe().T

## Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['aqi'], kde=True, ax=axes[0])
axes[0].set_title('AQI Distribution')
sns.boxplot(x=df['aqi'], ax=axes[1])
axes[1].set_title('AQI Boxplot (outlier check)')
plt.tight_layout(); plt.show()

print('AQI skewness:', df['aqi'].skew())
print('AQI kurtosis:', df['aqi'].kurtosis())

In [ ]:
pollutants = [c for c in ['pm2_5', 'pm10', 'co', 'no', 'no2', 'o3', 'so2', 'nh3'] if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, pollutants):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} (skew={df[col].skew():.2f})')
for ax in axes.flat[len(pollutants):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

print('Pollutant skewness:')
print(df[pollutants].skew().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(df['timestamp'], df['aqi'], linewidth=0.5)
plt.title('AQI Over Time — Lahore (raw)')
plt.xlabel('Date'); plt.ylabel('AQI')
plt.show()

# Yearly means — trend check (known multi-year decline)
yearly = df.set_index('timestamp')['aqi'].resample('YE').mean()
print('Yearly mean AQI:')
print(yearly)

## Bivariate analysis

In [ ]:
weather_cols = [c for c in ['temperature', 'humidity', 'wind_speed', 'pressure'] if c in df.columns]
fig, axes = plt.subplots(1, len(weather_cols), figsize=(18, 4))
for ax, col in zip(axes, weather_cols):
    sns.scatterplot(x=df[col], y=df['aqi'], alpha=0.15, ax=ax, s=8)
    ax.set_title(f'AQI vs {col} (r={df[col].corr(df["aqi"]):.2f})')
plt.tight_layout(); plt.show()

## Multivariate analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['month'], errors='ignore')
plt.figure(figsize=(12, 9))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap — Raw Columns')
plt.show()

print('Top correlations with aqi:')
print(numeric_df.corr()['aqi'].sort_values(ascending=False))

## Time-series analysis

In [ ]:
result = adfuller(df['aqi'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print('=> Stationary (reject H0)' if result[1] < 0.05 else '=> Non-stationary (fail to reject H0) — consider differencing / delta targets')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df['aqi'].dropna(), lags=168, ax=axes[0])
plot_pacf(df['aqi'].dropna(), lags=72, ax=axes[1], method='ywm')
axes[0].set_title('ACF (lags up to 168h)')
axes[1].set_title('PACF (lags up to 72h)')
plt.tight_layout(); plt.show()
# Significant spikes justify WHICH lag features to keep (e.g. lag 24 / 168).

In [ ]:
ts = df.set_index('timestamp')['aqi'].asfreq('h').interpolate()
decomposition = seasonal_decompose(ts, model='additive', period=24)
fig = decomposition.plot()
fig.set_size_inches(14, 8)
plt.tight_layout(); plt.show()

## Smog season vs. normal season

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['is_smog_season'].map({0: 'Normal', 1: 'Smog (Oct-Jan)'}), y=df['aqi'])
plt.title('AQI: Smog Season vs. Normal Season')
plt.xlabel(''); plt.ylabel('AQI')
plt.show()

print(df.groupby('is_smog_season')['aqi'].describe())

## Findings for FE


In [ ]:
# Auto-summary helpers — paste key numbers into the markdown table above
print('=== Auto findings snapshot ===')
print('AQI skew:', round(df['aqi'].skew(), 3))
print('AQI mean/median/std:', round(df['aqi'].mean(), 1), round(df['aqi'].median(), 1), round(df['aqi'].std(), 1))
iqr = df['aqi'].quantile(0.75) - df['aqi'].quantile(0.25)
outlier_hi = df['aqi'].quantile(0.75) + 1.5 * iqr
print(f'AQI IQR outliers above {outlier_hi:.0f}:', int((df['aqi'] > outlier_hi).sum()))
print('\nPollutant skew:')
print(df[pollutants].skew().round(3).sort_values(ascending=False))
print('\nCorr with aqi:')
print(numeric_df.corr()['aqi'].sort_values(ascending=False).round(3))
print('\nADF p-value:', round(adfuller(df['aqi'].dropna())[1], 6))
print('\nYearly mean AQI:')
print(df.set_index('timestamp')['aqi'].resample('YE').mean().round(1))
print('\nSmog vs normal mean AQI:')
print(df.groupby('is_smog_season')['aqi'].mean().round(1))